In [ ]:
from utils import plot_mesh, extract_mesh_from_sdf
from siren import Siren
from data import get_fixed_sdf_dataset
import sys
import open3d as o3d
import torch.optim as optim
import torch
from torch.nn import MSELoss
from torch.utils.data import TensorDataset, DataLoader
import os
import numpy as np


device = "cpu" #"cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")


In [ ]:
bunny = o3d.data.BunnyMesh()
mesh = o3d.io.read_triangle_mesh(bunny.path)
mesh.compute_vertex_normals()

vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)

plot_mesh(vertices, triangles, title="Stanford Bunny", show_axes=False)

In [4]:
X, Y, center, scale, min_bound, max_bound, padding = get_fixed_sdf_dataset(mesh)
dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=16384, shuffle=True)
import intel_extension_for_pytorch as ipex

model = Siren(in_features=3, hidden_features=256, hidden_layers=2, out_features=1).to(device)
criterion = MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)

if device.type == "cpu":
    model, optimizer = ipex.optimize(model, optimizer=optimizer, dtype=torch.float32)

for epoch in range(101):
    model.train()
    epoch_loss = 0.0

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * xb.size(0)

    total_loss = epoch_loss / len(dataloader.dataset)
    scheduler.step(total_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Loss: {1e5 * total_loss:.4f}")

/home/tomasz/miniconda3/envs/gpu/lib/python3.12/site-packages/intel_extension_for_pytorch/transformers/optimize.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


OSError: libcudart.so.13: cannot open shared object file: No such file or directory

In [26]:
verts, faces = extract_mesh_from_sdf(
    model=model,
    device=device,
    min_bound=min_bound,
    max_bound=max_bound,
    center=center,
    scale=scale,
    grid_res=128,
    padding=padding,
)


plot_mesh(verts, faces, title="INR Reconstruction")

In [27]:
torch.save(model.state_dict(), 'model.pth')
half_state = {k: v.half() for k, v in model.state_dict().items()}
torch.save(half_state, 'model_half.pth')

original_kb = os.path.getsize(bunny.path)   / 1024
full_kb = os.path.getsize('model.pth')  / 1024
half_kb = os.path.getsize('model_half.pth') / 1024
n_params = sum(p.numel() for p in model.parameters())

print(f"{'--- Compression Comparison ---':^45}")
print(f"{'Source':<25} {'Size (KB)':>8}  {'vs Original':>10}")
print(f"{'-'*45}")
print(f"{'Original PLY':<25} {original_kb:>8.2f}")
print(f"{'INR float32':<25} {full_kb:>8.2f}  {original_kb/full_kb:>9.1f}x")
print(f"{'INR float16':<25} {half_kb:>8.2f}  {original_kb/half_kb:>9.1f}x")
print(f"{'-'*45}")
print(f"Parameters: {n_params:,}")

       --- Compression Comparison ---        
Source                    Size (KB)  vs Original
---------------------------------------------
Original PLY               2962.10
INR float32                 522.17        5.7x
INR float16                 262.99       11.3x
---------------------------------------------
Parameters: 132,865
